In [0]:
dbutils.widgets.text("catalog", "scl")
catalog = dbutils.widgets.get("catalog")

#dim_customer

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

dim_customer = (
    spark.table(f"{catalog}.silver.customer")
    .drop("source_path", "ingested_at")
    .withColumnRenamed("c_custkey", "customer_key")
    .withColumnRenamed("c_name", "customer_name")
    .withColumnRenamed("c_address", "address")
    .withColumnRenamed("c_nationkey", "nation_key")
    .withColumnRenamed("c_phone", "phone")
    .withColumnRenamed("c_acctbal", "account_balance")
    .withColumnRenamed("c_mktsegment", "market_segment")
    .withColumnRenamed("c_comment", "comment")
)

# Ensure one row per customer
dim_customer = (
    dim_customer
    .withColumn("rn", row_number().over(Window.partitionBy("customer_key").orderBy(col("customer_key"))))
    .filter(col("rn") == 1)
    .drop("rn")
)

dim_customer.write.mode("overwrite").saveAsTable(f"{catalog}.gold.dim_customer")

#dim_supplier

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

dim_supplier = (
    spark.table(f"{catalog}.silver.supplier")
    .drop("source_path", "ingested_at")
    .withColumnRenamed("s_suppkey", "supplier_key")
    .withColumnRenamed("s_name", "supplier_name")
    .withColumnRenamed("s_address", "address")
    .withColumnRenamed("s_nationkey", "nation_key")
    .withColumnRenamed("s_phone", "phone")
    .withColumnRenamed("s_acctbal", "account_balance")
    .withColumnRenamed("s_comment", "comment"))

dim_supplier = (
    dim_supplier
    .withColumn("rn", row_number().over(Window.partitionBy("supplier_key").orderBy(col("supplier_key"))))
    .filter(col("rn") == 1)
    .drop("rn")
)

dim_supplier.write.mode("overwrite").saveAsTable(f"{catalog}.gold.dim_supplier")

#dim_product

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

dim_product = (
    spark.table(f"{catalog}.silver.part")
    .drop("validation_error")
    .withColumnRenamed("p_partkey", "part_key")
    .withColumnRenamed("p_name", "part_name")
    .withColumnRenamed("p_mfgr", "manufacturer")
    .withColumnRenamed("p_brand", "brand")
    .withColumnRenamed("p_type", "part_type")
    .withColumnRenamed("p_size", "size")
    .withColumnRenamed("p_container", "container")
    .withColumnRenamed("p_retailprice", "retail_price")
    .withColumnRenamed("p_comment", "comment")
)

dim_product = (
    dim_product
    .withColumn("rn", row_number().over(Window.partitionBy("part_key").orderBy(col("part_key"))))
    .filter(col("rn") == 1)
    .drop("rn")
)

dim_product.write.mode("overwrite").saveAsTable(f"{catalog}.gold.dim_product")

#dim_geography

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

nation = (
    spark.table(f"{catalog}.silver.nation")
)

region = (
    spark.table(f"{catalog}.silver.region")
    .withColumnRenamed("r_regionkey", "region_key")
    .withColumnRenamed("r_name", "region_name")
    .withColumnRenamed("r_comment", "region_comment")
)

dim_geography = (
    nation
    .join(region, nation.n_regionkey == region.region_key, "left")
    .drop("n_regionkey")
    .withColumnRenamed("n_nationkey", "nation_key")
    .withColumnRenamed("n_name", "nation_name")
    .withColumnRenamed("n_comment", "nation_comment")
    .select("nation_key", "nation_name", "nation_comment", "region_key", "region_name", "region_comment")
)

# Ensure one row per nation
dim_geography = (
    dim_geography
    .withColumn("rn", row_number().over(Window.partitionBy("nation_key").orderBy(col("nation_key"))))
    .filter(col("rn") == 1)
    .drop("rn")
)

dim_geography.write.mode("overwrite").saveAsTable(f"{catalog}.gold.dim_geography")

#fact_sales

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

lineitem = spark.table(f"{catalog}.silver.lineitem")
orders = (
    spark.table(f"{catalog}.silver.orders")
)

fact_sales = (
    lineitem
    .join(orders, lineitem["l_orderkey"] == orders["o_orderkey"], "left")
    .withColumnRenamed("l_orderkey", "order_key")
    .withColumnRenamed("l_linenumber", "line_number")
    .withColumnRenamed("l_partkey", "part_key")
    .withColumnRenamed("l_suppkey", "supplier_key")
    .withColumnRenamed("o_custkey", "customer_key")
    .withColumnRenamed("l_quantity", "quantity")
    .withColumnRenamed("l_extendedprice", "extended_price")
    .withColumnRenamed("l_discount", "discount")
    .withColumnRenamed("l_tax", "tax")
    .withColumnRenamed("o_orderdate", "order_date")
    .withColumnRenamed("l_shipdate", "ship_date")
    .withColumnRenamed("l_commitdate", "commit_date")
    .withColumnRenamed("l_receiptdate", "receipt_date")
    .withColumnRenamed("o_orderstatus", "order_status")
    .withColumnRenamed("l_returnflag", "return_flag")
    .withColumnRenamed("l_linestatus", "line_status")
    .withColumnRenamed("o_orderpriority", "order_priority")
    .withColumnRenamed("o_totalprice", "order_total_price")
    .withColumnRenamed("o_shippriority", "ship_priority")
    .withColumnRenamed("o_clerk", "clerk")
    .withColumnRenamed("l_shipinstruct", "ship_instruct")
    .withColumnRenamed("l_shipmode", "ship_mode")
    .withColumnRenamed("o_comment", "order_comment")
    .withColumnRenamed("l_comment", "line_comment")
    .drop("o_orderkey")
    .withColumn("discount_amount", col("extended_price") * col("discount"))
    .withColumn("net_revenue", col("extended_price") * (lit(1) - col("discount")))
    .withColumn("tax_amount", col("net_revenue") * col("tax"))
    .withColumn("total_amount", col("net_revenue") * (lit(1) + col("tax")))
)

# Ensure one row per order line
fact_sales = (
    fact_sales
    .withColumn("rn", row_number().over(Window.partitionBy("order_key", "line_number").orderBy(col("order_key"), col("line_number"))))
    .filter(col("rn") == 1)
    .drop("rn")
)

fact_sales.write.mode("overwrite").saveAsTable(f"{catalog}.gold.fact_sales")

#vw_annual_sales_kpis

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.vw_annual_sales_kpis AS
SELECT
    YEAR(order_date) AS order_year,
    SUM(extended_price * (1 - discount)) AS net_revenue,
    COUNT(DISTINCT order_key) AS total_orders,
    COUNT(DISTINCT customer_key) AS active_customers,
    ROUND(SUM(extended_price * (1 - discount)) / COUNT(DISTINCT order_key), 2) AS avg_order_value
FROM {catalog}.gold.fact_sales
GROUP BY YEAR(order_date)
ORDER BY order_year
""")

display(spark.table(f"{catalog}.gold.vw_annual_sales_kpis"))

#report

In [0]:
#gold_and_silver_reconcialtion

In [0]:
# CHECK 5 - Gold revenue and counts reconcile with silver

dq_results = []
recon_rows = []
#fact_sales -- silver.lineitem
gold_fact_rows = spark.table(f"{catalog}.gold.fact_sales").count()
silver_line_rows = spark.table(f"{catalog}.silver.lineitem").count()
status = "PASS" if gold_fact_rows == silver_line_rows else "FAIL"

recon_rows.append(("fact_sales rows vs silver.lineitem rows", float(silver_line_rows), float(gold_fact_rows), float(silver_line_rows - gold_fact_rows), status))

#rem tables
dq_results.append(("5. Recon - fact_sales vs lineitem rows", f"gold={gold_fact_rows:,} vs silver={silver_line_rows:,}", status))
dim_recon = [
    ("dim_customer", "customer"),
    ("dim_supplier", "supplier"),
    ("dim_product", "part"),
    ("dim_geography", "nation"),
]
for dim, src in dim_recon:
    dim_cnt = spark.table(f"{catalog}.gold.{dim}").count()
    src_cnt = spark.table(f"{catalog}.silver.{src}").count()
    status = "PASS" if dim_cnt == src_cnt else "FAIL"
    recon_rows.append((f"gold.{dim} rows vs silver.{src} rows", float(src_cnt), float(dim_cnt), float(src_cnt - dim_cnt), status))
    dq_results.append((f"5. Recon - {dim} vs silver.{src}", f"gold={dim_cnt:,} vs silver={src_cnt:,}", status))

display(
    spark.createDataFrame(recon_rows, ["reconciliation", "silver_or_source_value", "gold_value", "difference", "status"])
)

n_pass = sum(1 for r in recon_rows if r[4] == "PASS")
print(f"Reconciliation check: {n_pass}/{len(recon_rows)} reconciliations passed")